# KRYPTO LENS — Google Colab Notebook

A structured, runnable demo containing:
- A React-based frontend served by Express
- A Python heuristic engine for URL, phishing-message, email, and password checks
- An Express API with rate limiting
- Local JSON persistence (no MongoDB download or Atlas setup required)
- Optional Google Safe Browsing and VirusTotal lookups

**Run every code cell from top to bottom.** This is a learning/demo tool, not a replacement for a commercial threat-intelligence service. Heuristic results can produce false positives and false negatives.

**Privacy:** submitted passwords are evaluated in memory and are never written to the local data file. Avoid entering real passwords or confidential messages into a demo application.


## Upgrade notes
This notebook preserves the original Python analyzer, Express API, React single-page frontend, password checker, IP lookup, reporting and dashboard. Added modules include QR image/camera decoding (browser `jsQR`), multilingual interface labels (English/Kannada/Hindi), browser speech synthesis/recognition, PDF threat reports and scan-type dashboard counts.

**Privacy and safety:** QR images are decoded in the browser and are not uploaded. The extracted text/URL is sent to the existing analyzer API. The app never navigates to an extracted URL and does not follow redirects. Scan history stores a shortened preview and indicators, not the uploaded image. Avoid scanning highly sensitive content on a shared Colab runtime.


## 1. Prepare the Colab environment
Checks Node.js and Python and creates the project folders. Browser QR decoding and PDF generation use CDN-hosted `jsQR` and `jsPDF`; no extra Python package is required.

In [38]:
import os, shutil, subprocess, sys

if shutil.which("node") is None:
    subprocess.run(
        "curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && apt-get install -y nodejs",
        shell=True, check=True
    )

print("Node:", subprocess.check_output(["node", "-v"], text=True).strip())
print("npm:", subprocess.check_output(["npm", "-v"], text=True).strip())
print("Python:", sys.version.split()[0])

os.makedirs("cybershield/public", exist_ok=True)
os.makedirs("cybershield/data", exist_ok=True)
print("Project folders are ready.")


Node: v22.23.4
npm: 10.8.2
Python: 3.13.18
Project folders are ready.


## 2. Create the KRYPTO LENS Node.js package file

In [39]:
%%writefile cybershield/package.json
{
  "name": "krypto-lens",
  "version": "1.1.0",
  "private": true,
  "description": "KRYPTO LENS security awareness and heuristic scanning demo",
  "main": "server.js",
  "scripts": {
    "start": "node server.js"
  },
  "engines": {
    "node": ">=18"
  },
  "dependencies": {
    "express": "^4.21.2",
    "express-rate-limit": "^7.5.0",
    "helmet": "^7.2.0"
  }
}


Overwriting cybershield/package.json


## 3. Preserve the existing Python security-analysis engine

In [48]:
%%writefile cybershield/analyzer.py
import sys, json, re, math
from urllib.parse import urlparse

COMMON = {"password","123456","12345678","qwerty","abc123","letmein","admin","welcome","iloveyou","monkey","dragon","111111","passw0rd","password1","123456789"}
KW = ["login","verify","secure","account","update","bank","paypal","signin","confirm","wallet","password","gift","free"]
BAD_TLD = (".zip",".xyz",".top",".tk",".click",".gq",".ml",".work",".country")
SHORT = {"bit.ly","tinyurl.com","t.co","goo.gl","is.gd","cutt.ly","rb.gy"}
URG = ["urgent","immediately","act now","suspended","verify your","confirm your","unusual activity","limited time","click here","password expires","final notice","you have won","claim your","security alert"]
ASK = ["password","ssn","social security","credit card","otp","pin","bank account","cvv","gift card","wire transfer"]
BAD_EXT = (".exe",".scr",".js",".vbs",".iso",".docm",".xlsm",".bat",".jar",".html")


def verdict(s):
    return "Dangerous" if s >= 60 else "Suspicious" if s >= 30 else "Safe"


def url_scan(u):
    flags, score = [], 0
    u = u.strip()
    p = urlparse(u if "://" in u else "http://" + u)
    host = (p.hostname or "").lower()
    if not host:
        return {"score": 100, "verdict": "Dangerous", "flags": ["Not a valid URL"]}
    checks = [
        (p.scheme != "https", 15, "Connection is not encrypted (no HTTPS)"),
        (bool(re.fullmatch(r"[\d.]+", host)), 30, "Uses a raw IP address instead of a domain"),
        ("@" in u, 25, "Contains '@' which can hide the real destination"),
        (host.count(".") >= 4, 15, "Unusually many subdomains"),
        (host.startswith("xn--") or "xn--" in host, 25, "Punycode domain (look-alike characters)"),
        (len(u) > 90, 10, "Very long URL"),
        (host.endswith(BAD_TLD), 20, "High-risk top-level domain"),
        (host in SHORT, 15, "URL shortener hides the final destination"),
        (host.count("-") >= 3, 10, "Many hyphens in domain"),
    ]
    hits = [k for k in KW if k in (host + p.path).lower()]
    if len(hits) >= 2:
        checks.append((True, 20, "Sensitive keywords in URL: " + ", ".join(hits[:4])))
    for cond, pts, msg in checks:
        if cond:
            score += pts
            flags.append(msg)
    score = min(score, 100)
    if not flags:
        flags.append("No suspicious patterns found")
    return {"score": score, "verdict": verdict(score), "flags": flags}


def text_scan(t, email=False):
    low, flags, score = t.lower(), [], 0
    urg = [k for k in URG if k in low]
    ask = [k for k in ASK if k in low]
    if urg:
        score += min(15 * len(urg), 35)
        flags.append("Urgency / pressure wording: " + ", ".join(urg[:4]))
    if ask:
        score += min(15 * len(ask), 35)
        flags.append("Asks for sensitive data: " + ", ".join(ask[:4]))
    links = re.findall(r"https?://[^\s<>\"']+", t)
    worst = 0
    for l in links[:5]:
        r = url_scan(l)
        worst = max(worst, r["score"])
    if links:
        flags.append(f"{len(links)} link(s) found, riskiest scored {worst}/100")
        score += worst // 3
    if email:
        frm = re.search(r"^from:.*?<?([\w.+-]+@[\w.-]+)>?", t, re.I | re.M)
        rep = re.search(r"^reply-to:.*?<?([\w.+-]+@[\w.-]+)>?", t, re.I | re.M)
        if frm and rep and frm.group(1).split("@")[1] != rep.group(1).split("@")[1]:
            score += 25
            flags.append("From and Reply-To domains do not match")
        if any(e in low for e in BAD_EXT) and "attach" in low:
            score += 20
            flags.append("Mentions a risky attachment type")
        if re.search(r"dear (customer|user|member)", low):
            score += 10
            flags.append("Generic greeting instead of your name")
    if not flags:
        flags.append("No suspicious patterns found")
    score = min(score, 100)
    return {"score": score, "verdict": verdict(score), "flags": flags}


def pw_scan(p):
    flags, pool = [], 0
    for rx, n in ((r"[a-z]", 26), (r"[A-Z]", 26), (r"\d", 10), (r"[^\w]", 32)):
        if re.search(rx, p):
            pool += n
    ent = round(len(p) * math.log2(pool), 1) if pool else 0
    score = min(int(ent / 80 * 100), 100)
    if len(p) < 12:
        flags.append("Use at least 12 characters")
    if p.lower() in COMMON or any(c in p.lower() for c in ("password", "qwerty", "admin")):
        score = min(score, 10)
        flags.append("Contains a very common password pattern")
    if re.search(r"(.)\1{2,}", p):
        score -= 15
        flags.append("Repeated characters (aaa, 111)")
    if re.search(r"(0123|1234|2345|abcd|bcde)", p.lower()):
        score -= 15
        flags.append("Sequential characters")
    for rx, msg in ((r"[a-z]", "Add lowercase letters"), (r"[A-Z]", "Add uppercase letters"), (r"\d", "Add numbers"), (r"[^\w]", "Add symbols")):
        if not re.search(rx, p):
            flags.append(msg)
    score = max(0, min(score, 100))
    label = "Strong" if score >= 75 else "Medium" if score >= 45 else "Weak"
    if not flags:
        flags.append("Great password. Store it in a password manager.")
    return {"score": score, "verdict": label, "flags": flags, "entropy": ent}


if __name__ == "__main__":
    try:
        d = json.load(sys.stdin)
        t = str(d.get("type", ""))
        x = str(d.get("data", ""))
        fn = {
            "url": url_scan,
            "phishing": text_scan,
            "password": pw_scan,
            "email": lambda s: text_scan(s, True),
            "qr": text_scan
        }.get(t)
        if fn is None:
            # Fallback if unknown scan type passed
            fn = text_scan
        print(json.dumps(fn(x)))
    except Exception as exc:
        print(json.dumps({"error": str(exc)}))
        sys.exit(1)

Overwriting cybershield/analyzer.py


## 4. Express backend with QR analysis endpoint and dashboard counters

In [51]:
%%writefile cybershield/server.js
const express = require('express');
const helmet = require('helmet');
const rateLimit = require('express-rate-limit');
const path = require('path');
const fs = require('fs');
const { spawn } = require('child_process');

const app = express();
const PORT = Number(process.env.PORT || 3000);
const PUBLIC_DIR = path.join(__dirname, 'public');
const DATA_DIR = path.join(__dirname, 'data');
const DB_FILE = path.join(DATA_DIR, 'cybershield-data.json');

fs.mkdirSync(DATA_DIR, { recursive: true });
if (!fs.existsSync(DB_FILE)) {
  fs.writeFileSync(DB_FILE, JSON.stringify({ scans: [], reports: [] }, null, 2));
}

function readDB() {
  try {
    const parsed = JSON.parse(fs.readFileSync(DB_FILE, 'utf8'));
    return {
      scans: Array.isArray(parsed.scans) ? parsed.scans : [],
      reports: Array.isArray(parsed.reports) ? parsed.reports : []
    };
  } catch {
    return { scans: [], reports: [] };
  }
}
function writeDB(db) {
  fs.writeFileSync(DB_FILE, JSON.stringify(db, null, 2));
}

app.disable('x-powered-by');
app.use(helmet({
  contentSecurityPolicy: false,
  crossOriginEmbedderPolicy: false,
  frameguard: false
}));
app.use(express.json({ limit: '100kb' }));
app.use('/api', rateLimit({
  windowMs: 60 * 1000,
  limit: 100,
  standardHeaders: 'draft-7',
  legacyHeaders: false
}));

function analyze(type, data) {
  return new Promise((resolve, reject) => {
    const child = spawn('python3', [path.join(__dirname, 'analyzer.py')]);
    let stdout = '';
    let stderr = '';
    child.stdout.on('data', chunk => stdout += chunk.toString());
    child.stderr.on('data', chunk => stderr += chunk.toString());
    child.on('error', reject);
    child.on('close', code => {
      try {
        const result = JSON.parse(stdout);
        if (code !== 0 || result.error) {
          return reject(new Error(result.error || stderr || 'Analyzer failed'));
        }
        resolve(result);
      } catch {
        reject(new Error(stderr || 'Could not parse analyzer response'));
      }
    });
    child.stdin.end(JSON.stringify({ type, data }));
  });
}

async function safeBrowsing(url) {
  if (!process.env.GSB_KEY) return null;
  const response = await fetch(
    `https://safebrowsing.googleapis.com/v4/threatMatches:find?key=${encodeURIComponent(process.env.GSB_KEY)}`,
    {
      method: 'POST',
      headers: { 'Content-Type': 'application/json' },
      body: JSON.stringify({
        client: { clientId: 'cybershield-demo', clientVersion: '1.1' },
        threatInfo: {
          threatTypes: ['MALWARE', 'SOCIAL_ENGINEERING', 'UNWANTED_SOFTWARE'],
          platformTypes: ['ANY_PLATFORM'],
          threatEntryTypes: ['URL'],
          threatEntries: [{ url }]
        }
      })
    }
  );
  if (!response.ok) return null;
  const body = await response.json();
  return body.matches?.length ? 'Google Safe Browsing reported a threat' : null;
}

async function virusTotal(url) {
  if (!process.env.VT_KEY) return null;
  // VirusTotal URL identifiers use URL-safe base64 without padding.
  const id = Buffer.from(url).toString('base64').replace(/=+$/g, '')
    .replace(/\+/g, '-').replace(/\//g, '_');
  const response = await fetch(`https://www.virustotal.com/api/v3/urls/${id}`, {
    headers: { 'x-apikey': process.env.VT_KEY }
  });
  if (!response.ok) return null;
  const body = await response.json();
  const stats = body.data?.attributes?.last_analysis_stats;
  if (!stats) return null;
  const malicious = Number(stats.malicious || 0);
  const suspicious = Number(stats.suspicious || 0);
  return malicious + suspicious > 0
    ? `VirusTotal: ${malicious} malicious, ${suspicious} suspicious detections`
    : null;
}

// Specific routes MUST be declared BEFORE generic parameter routes like /api/scan/:type
// QR analysis reuses the existing Python URL/message analyzer. QR contents are
// never opened or fetched; URL redirects are not followed by this application.
app.post('/api/scan/qr', async (req, res) => {
  try {
    const content = String(req.body?.data ?? '').trim().slice(0, 20000);
    if (!content) return res.status(400).json({ error: 'No QR content was decoded' });
    const looksLikeUrl = /^(https?:\/\/|www\.)/i.test(content) || /^[a-z0-9.-]+\.[a-z]{2,}(\/|$)/i.test(content);
    const scanType = looksLikeUrl ? 'url' : 'phishing';
    const result = await analyze(scanType, content);
    result.qrContentType = looksLikeUrl ? 'URL' : 'Text';
    result.extractedContent = content.slice(0, 500);
    result.sources = ['CyberShield heuristic checks (QR content)'];
    if (looksLikeUrl) {
      const hits = (await Promise.allSettled([safeBrowsing(content), virusTotal(content)]))
        .filter(x => x.status === 'fulfilled' && x.value).map(x => x.value);
      if (hits.length) {
        result.score = Math.max(Number(result.score) || 0, 90);
        result.verdict = 'Dangerous';
        result.flags = [...hits, ...(result.flags || [])];
        result.sources.push('External reputation services');
      }
    }
    const db = readDB();
    db.scans.unshift({
      id: `${Date.now()}-${Math.random().toString(16).slice(2)}`,
      type: 'qr', input: `QR ${result.qrContentType}: ${content.slice(0, 160)}`,
      score: result.score, risk: result.score, verdict: result.verdict,
      flags: result.flags || [], createdAt: new Date().toISOString()
    });
    db.scans = db.scans.slice(0, 500);
    writeDB(db);
    res.json(result);
  } catch (error) {
    console.error('QR scan error:', error.message);
    res.status(500).json({ error: 'QR analysis failed' });
  }
});

app.post('/api/scan/:type', async (req, res) => {
  try {
    const type = req.params.type;
    const data = String(req.body?.data ?? '').trim().slice(0, 20000);
    if (!['url', 'phishing', 'email', 'password', 'qr'].includes(type)) {
      return res.status(400).json({ error: 'Unsupported scan type' });
    }
    if (!data) return res.status(400).json({ error: 'Please enter something to scan' });

    const result = await analyze(type, data);
    result.sources = ['CyberShield heuristic checks'];

    if (type === 'url') {
      const reputationResults = await Promise.allSettled([
        safeBrowsing(data), virusTotal(data)
      ]);
      const hits = reputationResults
        .filter(item => item.status === 'fulfilled' && item.value)
        .map(item => item.value);
      if (hits.length) {
        result.score = Math.max(Number(result.score) || 0, 90);
        result.verdict = 'Dangerous';
        result.flags = [...hits, ...(result.flags || [])];
        result.sources.push('External reputation services');
      }
    }

    const risk = type === 'password' ? 100 - result.score : result.score;
    // Never persist the submitted password or message body.
    const savedInput = type === 'password'
      ? `Password not stored (${data.length} characters)`
      : data.slice(0, 200);
    const db = readDB();
    db.scans.unshift({
      id: `${Date.now()}-${Math.random().toString(16).slice(2)}`,
      type,
      input: savedInput,
      score: result.score,
      risk,
      verdict: result.verdict,
      flags: result.flags || [],
      createdAt: new Date().toISOString()
    });
    db.scans = db.scans.slice(0, 500);
    writeDB(db);
    res.json(result);
  } catch (error) {
    console.error('Scan error:', error.message);
    res.status(500).json({ error: 'Scan failed. Check the server log for details.' });
  }
});

app.get('/api/dashboard', (_req, res) => {
  const db = readDB();
  const history = db.scans.slice(0, 10);
  const alerts = db.scans
    .filter(scan => scan.risk >= 30 && scan.type !== 'password')
    .slice(0, 6);
  const recent = db.scans.slice(0, 20);
  const averageRisk = recent.length
    ? recent.reduce((sum, scan) => sum + (Number(scan.risk) || 0), 0) / recent.length
    : 0;
  res.json({
    history,
    alerts,
    score: Math.round(100 - averageRisk),
    stats: {
      total: db.scans.length,
      emails: db.scans.filter(s => s.type === 'email' || s.type === 'phishing').length,
      urls: db.scans.filter(s => s.type === 'url').length,
      qrs: db.scans.filter(s => s.type === 'qr').length,
      safe: db.scans.filter(s => s.verdict === 'Safe').length,
      suspicious: db.scans.filter(s => s.verdict === 'Suspicious').length,
      dangerous: db.scans.filter(s => s.verdict === 'Dangerous').length,
      threats: db.scans.filter(scan => scan.verdict === 'Dangerous').length,
      reports: db.reports.length
    }
  });
});

app.get('/api/ip', async (_req, res) => lookupIP('', res));
app.get('/api/ip/:ip', async (req, res) => lookupIP(req.params.ip, res));

async function lookupIP(ip, res) {
  const net = require('net');
  if (ip && !net.isIP(ip)) {
    return res.status(400).json({ error: 'Enter a valid IPv4 or IPv6 address' });
  }
  try {
    const response = await fetch(`https://ipwho.is/${encodeURIComponent(ip)}`);
    if (!response.ok) throw new Error('IP lookup service returned an error');
    const body = await response.json();
    if (!body.success) {
      return res.status(404).json({ error: body.message || 'IP lookup failed' });
    }
    res.json({
      ip: body.ip,
      type: body.type,
      country: body.country,
      region: body.region,
      city: body.city,
      isp: body.connection?.isp || null,
      org: body.connection?.org || null,
      asn: body.connection?.asn || null,
      timezone: body.timezone?.id || null
    });
  } catch (error) {
    console.error('IP lookup error:', error.message);
    res.status(502).json({ error: 'IP lookup service is currently unavailable' });
  }
}

app.post('/api/report', (req, res) => {
  const url = String(req.body?.url ?? '').trim().slice(0, 500);
  const note = String(req.body?.note ?? '').trim().slice(0, 1000);
  if (!url) return res.status(400).json({ error: 'URL is required' });
  const db = readDB();
  db.reports.unshift({
    id: `${Date.now()}-${Math.random().toString(16).slice(2)}`,
    url,
    note,
    createdAt: new Date().toISOString()
  });
  db.reports = db.reports.slice(0, 500);
  writeDB(db);
  res.json({ ok: true });
});

app.use(express.static(PUBLIC_DIR));
app.get('*', (_req, res) => res.sendFile(path.join(PUBLIC_DIR, 'index.html')));

app.use((err, _req, res, _next) => {
  console.error('Unhandled server error:', err);
  res.status(500).json({ error: 'Unexpected server error' });
});

app.listen(PORT, '0.0.0.0', () => {
  console.log(`KRYPTO LENS is running on port ${PORT}`);
  console.log('Local demo storage: cybershield/data/cybershield-data.json');
  console.log('Optional threat-intelligence APIs:', Boolean(process.env.GSB_KEY || process.env.VT_KEY));
});


Overwriting cybershield/server.js


## 5. React frontend: KRYPTO LENS, QR, languages and voice tools

In [42]:
%%writefile cybershield/public/index.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>KRYPTO LENS</title>
<script src="https://cdn.jsdelivr.net/npm/jsqr@1.4.0/dist/jsQR.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/jspdf/2.5.1/jspdf.umd.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/react/18.3.1/umd/react.production.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/react-dom/18.3.1/umd/react-dom.production.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/babel-standalone/7.24.7/babel.min.js"></script>
<style>
:root{--bg:#070d1a;--panel:#0f1a2e;--line:#1e2d4a;--text:#e6edf7;--muted:#8fa3c0;--cyan:#22d3ee;--green:#34d399;--amber:#fbbf24;--red:#f87171}
*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--text);font:15px/1.55 system-ui,-apple-system,Segoe UI,Roboto,sans-serif}
button,input,textarea{font:inherit}
nav{position:sticky;top:0;z-index:5;display:flex;gap:6px;align-items:center;flex-wrap:wrap;padding:12px 24px;background:#070d1aee;border-bottom:1px solid var(--line);backdrop-filter:blur(8px)}
.logo{font-weight:800;font-size:18px;margin-right:auto;color:var(--cyan)}
nav button{background:none;border:0;color:var(--muted);padding:8px 12px;border-radius:8px;cursor:pointer}
nav button:hover,nav button.on{color:var(--text);background:var(--panel)}
button:focus-visible,input:focus-visible,textarea:focus-visible{outline:2px solid var(--cyan);outline-offset:2px}
.hero{min-height:78vh;display:grid;place-items:center;text-align:center;padding:40px 24px;
background:linear-gradient(180deg,#070d1acc,#070d1af2),url(https://images.unsplash.com/photo-1550751827-4bd374c3f58b?auto=format&fit=crop&w=1800&q=70) center/cover,radial-gradient(circle at 50% 30%,#0e3b4f,#070d1a)}
.hero h1{font-size:clamp(34px,6vw,64px);margin:0 0 12px;line-height:1.1}
.hero p{max-width:620px;margin:0 auto 26px;color:var(--muted);font-size:18px}
.btn{background:var(--cyan);color:#04202a;border:0;padding:11px 20px;border-radius:10px;font-weight:700;cursor:pointer}
.btn.alt{background:transparent;color:var(--text);border:1px solid var(--line);margin-left:10px}
.btn:disabled{opacity:.5;cursor:wait}
main{max-width:1100px;margin:0 auto;padding:28px 20px 60px}
h2{margin:0 0 4px;font-size:26px}.sub{color:var(--muted);margin:0 0 22px}
.grid{display:grid;gap:16px;grid-template-columns:repeat(auto-fit,minmax(240px,1fr))}
.card{background:var(--panel);border:1px solid var(--line);border-radius:14px;padding:18px}
.card h3{margin:0 0 6px;font-size:16px}.card p{margin:0;color:var(--muted)}
.feat{cursor:pointer;transition:border-color .15s}.feat:hover{border-color:var(--cyan)}
input,textarea{width:100%;background:#0a1324;color:var(--text);border:1px solid var(--line);border-radius:10px;padding:11px 12px;margin:8px 0 12px}
textarea{min-height:150px;resize:vertical}
.tabs{display:flex;gap:8px;margin-bottom:14px;flex-wrap:wrap}
.tabs button{background:var(--panel);color:var(--muted);border:1px solid var(--line);padding:8px 14px;border-radius:999px;cursor:pointer}
.tabs button.on{color:#04202a;background:var(--cyan);border-color:var(--cyan)}
.bar{height:10px;background:#0a1324;border-radius:99px;overflow:hidden;margin:10px 0}.bar i{display:block;height:100%;transition:width .5s}
.pill{display:inline-block;padding:2px 10px;border-radius:99px;font-size:12px;font-weight:700}
.Safe,.Strong{background:#34d39922;color:var(--green)}.Suspicious,.Medium{background:#fbbf2422;color:var(--amber)}.Dangerous,.Weak{background:#f8717122;color:var(--red)}
ul.fl{margin:8px 0 0;padding-left:18px;color:var(--muted)}
table{width:100%;border-collapse:collapse}th,td{text-align:left;padding:9px 8px;border-bottom:1px solid var(--line);font-size:14px}
th{color:var(--muted);font-weight:600}td.in{max-width:260px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
.ring{width:150px;height:150px;border-radius:50%;display:grid;place-items:center;margin:6px auto}
.ring div{width:114px;height:114px;border-radius:50%;background:var(--panel);display:grid;place-items:center;font-size:36px;font-weight:800}
.stat{font-size:30px;font-weight:800}.two{display:grid;gap:16px;grid-template-columns:1fr 2fr}
.kv{display:grid;grid-template-columns:130px 1fr;gap:6px;padding:7px 0;border-bottom:1px solid var(--line)}.kv span{color:var(--muted)}
.alert{padding:10px 0;border-bottom:1px solid var(--line)}.err{color:var(--red)}.ok{color:var(--green)}
@media(max-width:760px){.two{grid-template-columns:1fr}}
@media(prefers-reduced-motion:reduce){*{transition:none!important}}
</style>
</head>
<body>
<div id="root"></div>
<script type="text/babel">
const { useState, useEffect } = React;
const api = async (p, body) => {
  try {
    const response = await fetch('/api/' + p, {
      method: body ? 'POST' : 'GET',
      headers: { 'Content-Type': 'application/json' },
      ...(body ? { body: JSON.stringify(body) } : {})
    });
    const data = await response.json();
    if (!response.ok) throw new Error(data.error || 'Request failed');
    return data;
  } catch (error) {
    return { error: error.message || 'Network error' };
  }
};
const color = s => s >= 60 ? 'var(--red)' : s >= 30 ? 'var(--amber)' : 'var(--green)';

const ago = d => new Date(d).toLocaleString();
const I18N = {
 en:{Home:'Home',Dashboard:'Dashboard','Threat Scanner':'Threat Scanner','QR Analyzer':'QR Analyzer',Password:'Password','IP Lookup':'IP Lookup',Report:'Report',Awareness:'Awareness','Threat scanner':'Threat scanner','Scan now':'Scan now','Scanning…':'Scanning…','Security dashboard':'Security dashboard','Security score':'Security score','Recent alerts':'Recent alerts','Scan history':'Scan history','Password strength':'Password strength','IP lookup':'IP lookup','Look up':'Look up','Report a threat':'Report a threat','Send report':'Send report','Cyber awareness':'Cyber awareness','Open dashboard':'Open dashboard','Scan a link':'Scan a link','Safe':'Safe','Suspicious':'Suspicious','Dangerous':'Dangerous','Play':'Play','Pause':'Pause','Stop':'Stop','Voice guidance':'Voice guidance','Download PDF report':'Download PDF report','Upload QR image':'Upload QR image','Start camera scan':'Start camera scan','Stop camera':'Stop camera','Analyze QR':'Analyze QR','Enable voice assistance':'Enable voice assistance','Ask by voice':'Ask by voice','Listen':'Listen'},
 hi:{Home:'होम',Dashboard:'डैशबोर्ड','Threat Scanner':'खतरा स्कैनर','QR Analyzer':'QR विश्लेषक',Password:'पासवर्ड','IP Lookup':'IP लुकअप',Report:'रिपोर्ट',Awareness:'जागरूकता','Threat scanner':'खतरा स्कैनर','Scan now':'अभी स्कैन करें','Scanning…':'स्कैन हो रहा है…','Security dashboard':'सुरक्षा डैशबोर्ड','Security score':'सुरक्षा स्कोर','Recent alerts':'हाल के अलर्ट','Scan history':'स्कैन इतिहास','Password strength':'पासवर्ड की मजबूती','IP lookup':'IP लुकअप','Look up':'खोजें','Report a threat':'खतरे की रिपोर्ट करें','Send report':'रिपोर्ट भेजें','Cyber awareness':'साइबर जागरूकता','Open dashboard':'डैशबोर्ड खोलें','Scan a link':'लिंक स्कैन करें','Safe':'सुरक्षित','Suspicious':'संदिग्ध','Dangerous':'खतरनाक','Play':'चलाएं','Pause':'रोकें','Stop':'बंद करें','Voice guidance':'आवाज़ मार्गदर्शन','Download PDF report':'PDF रिपोर्ट डाउनलोड करें','Upload QR image':'QR छवि अपलोड करें','Start camera scan':'कैमरा स्कैन शुरू करें','Stop camera':'कैमरा बंद करें','Analyze QR':'QR का विश्लेषण करें','Enable voice assistance':'आवाज़ सहायता सक्षम करें','Ask by voice':'आवाज़ से पूछें','Listen':'सुनें'},
 kn:{Home:'ಮುಖಪುಟ',Dashboard:'ಡ್ಯಾಶ್‌ಬೋರ್ಡ್','Threat Scanner':'ಅಪಾಯ ಸ್ಕ್ಯಾನರ್','QR Analyzer':'QR ವಿಶ್ಲೇಷಕ',Password:'ಪಾಸ್‌ವರ್ಡ್','IP Lookup':'IP ಹುಡುಕಾಟ',Report:'ವರದಿ',Awareness:'ಜಾಗೃತಿ','Threat scanner':'ಅಪಾಯ ಸ್ಕ್ಯಾನರ್','Scan now':'ಈಗ ಸ್ಕ್ಯಾನ್ ಮಾಡಿ','Scanning…':'ಸ್ಕ್ಯಾನ್ ಆಗುತ್ತಿದೆ…','Security dashboard':'ಭದ್ರತಾ ಡ್ಯಾಶ್‌ಬೋರ್ಡ್','Security score':'ಭದ್ರತಾ ಅಂಕ','Recent alerts':'ಇತ್ತೀಚಿನ ಎಚ್ಚರಿಕೆಗಳು','Scan history':'ಸ್ಕ್ಯಾನ್ ಇತಿಹಾಸ','Password strength':'ಪಾಸ್‌ವರ್ಡ್ ಸಾಮರ್ಥ್ಯ','IP lookup':'IP ಹುಡುಕಾಟ','Look up':'ಹುಡುಕಿ','Report a threat':'ಅಪಾಯ ವರದಿ ಮಾಡಿ','Send report':'ವರದಿ ಕಳುಹಿಸಿ','Cyber awareness':'ಸೈಬರ್ ಜಾಗೃತಿ','Open dashboard':'ಡ್ಯಾಶ್‌ಬೋರ್ಡ್ ತೆರೆಯಿರಿ','Scan a link':'ಲಿಂಕ್ ಸ್ಕ್ಯಾನ್ ಮಾಡಿ','Safe':'ಸುರಕ್ಷಿತ','Suspicious':'ಸಂದೇಹಾಸ್ಪದ','Dangerous':'ಅಪಾಯಕಾರಿ','Play':'ಚಲಾಯಿಸಿ','Pause':'ವಿರಾಮ','Stop':'ನಿಲ್ಲಿಸಿ','Voice guidance':'ಧ್ವನಿ ಮಾರ್ಗದರ್ಶನ','Download PDF report':'PDF ವರದಿ ಡೌನ್‌ಲೋಡ್ ಮಾಡಿ','Upload QR image':'QR ಚಿತ್ರ ಅಪ್‌ಲೋಡ್ ಮಾಡಿ','Start camera scan':'ಕ್ಯಾಮೆರಾ ಸ್ಕ್ಯಾನ್ ಪ್ರಾರಂಭಿಸಿ','Stop camera':'ಕ್ಯಾಮೆರಾ ನಿಲ್ಲಿಸಿ','Analyze QR':'QR ವಿಶ್ಲೇಷಿಸಿ','Enable voice assistance':'ಧ್ವನಿ ಸಹಾಯ ಸಕ್ರಿಯಗೊಳಿಸಿ','Ask by voice':'ಧ್ವನಿಯಿಂದ ಕೇಳಿ','Listen':'ಕೇಳಿ'}
};
let currentLang='en', voiceEnabled=true, activeUtterance=null, lastScanResult=null;
const langLocale={en:'en-US',hi:'hi-IN',kn:'kn-IN'};
function speakText(text, mode='play'){
 if(!('speechSynthesis' in window)){alert('Text-to-Speech is not supported in this browser.');return;}
 if(mode==='pause'){speechSynthesis.pause();return;} if(mode==='stop'){speechSynthesis.cancel();return;}
 speechSynthesis.cancel(); activeUtterance=new SpeechSynthesisUtterance(text);
 activeUtterance.lang=langLocale[currentLang]||'en-US'; speechSynthesis.speak(activeUtterance);
}
function translatePage(){
 const dict=I18N[currentLang]||I18N.en;
 const walker=document.createTreeWalker(document.body,NodeFilter.SHOW_TEXT);
 let node; while((node=walker.nextNode())){
   if(!node.parentElement || ['SCRIPT','STYLE','TEXTAREA'].includes(node.parentElement.tagName)) continue;
   if(!node.dataset.krOriginal) node.dataset.krOriginal=node.nodeValue;
   const original=node.dataset.krOriginal.trim();
   if(dict[original]) node.nodeValue=node.nodeValue.replace(original,dict[original]);
 }
 document.documentElement.lang=currentLang;
}
function setLanguage(lang){currentLang=lang;translatePage();}
const translationObserver=new MutationObserver(()=>translatePage());
translationObserver.observe(document.body,{childList:true,subtree:true,characterData:true});


const PAGES = [['home','Home'],['dashboard','Dashboard'],['scanner','Threat Scanner'],['qr','QR Analyzer'],['password','Password'],['ip','IP Lookup'],['report','Report'],['awareness','Awareness']];

function Result({ r, strength }) {
  if (!r) return null;
  if (r.error) return <p className="err">{r.error}</p>;
  const c = strength ? color(100 - r.score) : color(r.score);
  return (
    <div className="card" style={{ marginTop: 16 }}>
      <span className={'pill ' + r.verdict}>{r.verdict}</span>{' '}
      <b>{strength ? `Strength ${r.score}/100` : `Risk ${r.score}/100`}</b>
      {r.entropy !== undefined && <span style={{ color: 'var(--muted)' }}> · {r.entropy} bits entropy</span>}
      <div className="bar"><i style={{ width: r.score + '%', background: c }} /></div>
      <ul className="fl">{(r.flags || []).map((f, i) => <li key={i}>{f}</li>)}</ul>
      <p style={{color:'var(--muted)',marginTop:10}}>{r.explanation || 'Review each indicator before interacting with this content.'}</p>
      <p><b>Recommended action:</b> {r.verdict==='Safe'?'Still verify the sender and destination.': 'Do not open the link or share passwords, OTPs or payment details. Verify through the official website or app.'}</p>
      <div className="tabs"><button onClick={()=>speakText(`${r.verdict}. Risk score ${r.score} out of 100. ${(r.flags||[]).join('. ')}. ${r.verdict==='Safe'?'Continue to stay alert.':'Do not open this link. Verify using the official website.'}`)}>▶ Play</button><button onClick={()=>speakText('', 'pause')}>Ⅱ Pause</button><button onClick={()=>speakText('', 'stop')}>■ Stop</button><button onClick={()=>downloadThreatPDF(r)}>Download PDF report</button></div>
    </div>
  );
}

function Home({ go }) {
  const f = [['scanner','URL Scanner','Checks suspicious website URLs'],['scanner','Phishing Detector','Identifies potential phishing messages'],['password','Password Strength Checker','Evaluates password strength'],['scanner','Email Threat Analyzer','Detects suspicious email content'],['ip','IP Lookup','Displays publicly available IP information'],['dashboard','Security Dashboard','Shows security statistics and alerts'],['awareness','Cyber Awareness','Provides cybersecurity tips'],['report','Report Threat','Allows users to report suspicious links']];
  return (<>
    <section className="hero"><div>
      <h1>Check it before you click it.</h1>
      <p>KRYPTO LENS scans links, messages and emails for threats, and tests how strong your passwords really are.</p>
      <button className="btn" onClick={() => go('scanner')}>Scan a link</button>
      <button className="btn alt" onClick={() => go('dashboard')}>Open dashboard</button>
    </div></section>
    <main><div className="grid">{f.map(([p, t, d]) =>
      <div key={t} className="card feat" tabIndex="0" onClick={() => go(p)} onKeyDown={e => e.key === 'Enter' && go(p)}><h3>{t}</h3><p>{d}</p></div>)}</div></main>
  </>);
}

function Dashboard() {
  const [d, setD] = useState(null);
  useEffect(() => {
    api('dashboard').then(data => setD(data.error ? { error: data.error } : data));
  }, []);
  if (!d) return <main><p>Loading…</p></main>;
  if (d.error) return <main><p className="err">{d.error}</p></main>;
  const c = color(100 - d.score);
  return (<main>
    <h2>Security dashboard</h2><p className="sub">Your score reflects your last 20 scans.</p>
    <div className="grid" style={{marginBottom:16}}>
      {[['Emails analyzed',d.stats.emails||0],['URLs analyzed',d.stats.urls||0],['QR codes analyzed',d.stats.qrs||0],['Safe',d.stats.safe||0],['Suspicious',d.stats.suspicious||0],['Dangerous',d.stats.dangerous||0]].map(([label,value])=><div className="card" key={label}><p>{label}</p><div className="stat">{value}</div></div>)}
    </div>
    <div className="two">
      <div className="card" style={{ textAlign: 'center' }}>
        <h3>Security score</h3>
        <div className="ring" style={{ background: `conic-gradient(${c} ${d.score * 3.6}deg,#0a1324 0)` }}><div>{d.score}</div></div>
        <p>{d.stats.total} scans · {d.stats.threats} dangerous · {d.stats.reports} reports</p>
      </div>
      <div className="card"><h3>Recent alerts</h3>
        {d.alerts.length === 0 && <p>No alerts. Scan a link or message to get started.</p>}
        {d.alerts.map(a => <div className="alert" key={a._id}><span className={'pill ' + a.verdict}>{a.verdict}</span> <b>{a.type}</b> · {a.input.slice(0, 60)}<br /><small style={{ color: 'var(--muted)' }}>{a.flags[0]} · {ago(a.createdAt)}</small></div>)}
      </div>
    </div>
    <div className="card" style={{ marginTop: 16, overflowX: 'auto' }}><h3>Scan history</h3>
      {d.history.length === 0 ? <p>No scans yet.</p> :
      <table><thead><tr><th>Type</th><th>Input</th><th>Result</th><th>Time</th></tr></thead><tbody>
        {d.history.map(h => <tr key={h._id}><td>{h.type}</td><td className="in">{h.input}</td><td><span className={'pill ' + h.verdict}>{h.verdict}</span></td><td>{ago(h.createdAt)}</td></tr>)}
      </tbody></table>}
    </div>
  </main>);
}

function Scanner() {
  const T = { url: ['URL Scanner', 'https://example.com/login', false], phishing: ['Phishing Detector', 'Paste the message text here', true], email: ['Email Threat Analyzer', 'Paste the full email, including From: and Reply-To: lines', true] };
  const [t, setT] = useState('url'), [v, setV] = useState(''), [r, setR] = useState(null), [busy, setBusy] = useState(false);
  const run = async () => {
    if (!v.trim() || busy) return;
    setBusy(true);
    setR(null);
    try {
      setR(await api('scan/' + t, { data: v.trim() }));
    } finally {
      setBusy(false);
    }
  };
  return (<main>
    <h2>Threat scanner</h2><p className="sub">Heuristic analysis plus Google Safe Browsing and VirusTotal when API keys are set.</p>
    <div className="tabs">{Object.entries(T).map(([k, x]) => <button key={k} className={t === k ? 'on' : ''} onClick={() => { setT(k); setR(null); setV(''); }}>{x[0]}</button>)}</div>
    <div className="card">
      {T[t][2] ? <textarea aria-label={T[t][0]} placeholder={T[t][1]} value={v} onChange={e => setV(e.target.value)} />
        : <input aria-label="URL" placeholder={T[t][1]} value={v} onChange={e => setV(e.target.value)} onKeyDown={e => e.key === 'Enter' && v && run()} />}
      <button className="btn" disabled={busy || !v.trim()} onClick={run}>{busy ? 'Scanning…' : 'Scan now'}</button>
      <button className="btn alt" onClick={()=>startVoiceQuestion(setV)}>🎙 Ask by voice</button>
    </div>
    <Result r={r} />
  </main>);
}


function startVoiceQuestion(setValue){
 const SR=window.SpeechRecognition||window.webkitSpeechRecognition;
 if(!SR){alert('Speech recognition is not supported in this browser. You can type your question instead.');return;}
 const rec=new SR(); rec.lang=langLocale[currentLang]||'en-US'; rec.interimResults=false;
 rec.onresult=e=>setValue(e.results[0][0].transcript);
 rec.onerror=e=>alert('Microphone/speech recognition issue: '+e.error+'. Check browser permissions or type instead.');
 rec.start();
}
function downloadThreatPDF(r){
 if(!window.jspdf?.jsPDF){alert('PDF library is unavailable. Check your internet connection and reload.');return;}
 const doc=new window.jspdf.jsPDF(); let y=18;
 doc.setFontSize(18);doc.text('KRYPTO LENS - Threat Analysis Report',14,y);y+=12;
 doc.setFontSize(11);const lines=[
  `Generated: ${new Date().toLocaleString()}`,`Risk score: ${r.score}/100`,`Risk category: ${r.verdict}`,
  `QR content type: ${r.qrContentType||'N/A'}`,`Analyzed content: ${(r.extractedContent||'').slice(0,180)}`,
  'Detection indicators:',...(r.flags||[]).map(x=>'• '+x),
  'Safety recommendation: Do not open suspicious links. Verify through the official website or app.'
 ];
 lines.forEach(line=>{const wrapped=doc.splitTextToSize(line,180);if(y+wrapped.length*6>280){doc.addPage();y=18;}doc.text(wrapped,14,y);y+=wrapped.length*6+2;});
 doc.save('krypto-lens-threat-report.pdf');
}
function QRAnalyzer(){
 const [result,setResult]=useState(null),[busy,setBusy]=useState(false),[camera,setCamera]=useState(false);
 const videoRef=React.useRef(null),streamRef=React.useRef(null),timerRef=React.useRef(null);
 const analyzeQR=async text=>{setBusy(true);try{const r=await api('scan/qr',{data:text});setResult(r);lastScanResult=r;}finally{setBusy(false);}};
 const decodeFile=e=>{
  const file=e.target.files?.[0];if(!file)return;
  if(!file.type.startsWith('image/')){alert('Please choose an image file.');return;}
  const img=new Image(),url=URL.createObjectURL(file);
  img.onload=()=>{try{const c=document.createElement('canvas');c.width=img.naturalWidth;c.height=img.naturalHeight;const ctx=c.getContext('2d');ctx.drawImage(img,0,0);const pixels=ctx.getImageData(0,0,c.width,c.height);const code=window.jsQR?.(pixels.data,pixels.width,pixels.height);if(code?.data)analyzeQR(code.data);else alert('No QR code found. Try a clearer, well-lit image.');}catch(err){alert('Could not decode this image: '+err.message);}finally{URL.revokeObjectURL(url);}};
  img.onerror=()=>{URL.revokeObjectURL(url);alert('Invalid or unsupported image.');};img.src=url;e.target.value='';
 };
 const startCamera=async()=>{
  if(!navigator.mediaDevices?.getUserMedia){alert('Camera access is unavailable. Upload a QR image instead.');return;}
  try{const s=await navigator.mediaDevices.getUserMedia({video:{facingMode:{ideal:'environment'}},audio:false});streamRef.current=s;setCamera(true);await new Promise(resolve=>setTimeout(resolve,150));if(videoRef.current){videoRef.current.srcObject=s;await videoRef.current.play();}timerRef.current=setInterval(()=>{const v=videoRef.current;if(!v||v.readyState<2||busy)return;const c=document.createElement('canvas');c.width=v.videoWidth;c.height=v.videoHeight;const x=c.getContext('2d');x.drawImage(v,0,0);const code=window.jsQR?.(x.getImageData(0,0,c.width,c.height).data,c.width,c.height);if(code?.data){stopCamera();analyzeQR(code.data);}},350);}catch(err){alert('Camera permission or device error: '+err.message+'. You can upload an image instead.');}
 };
 const stopCamera=()=>{if(timerRef.current)clearInterval(timerRef.current);streamRef.current?.getTracks().forEach(t=>t.stop());streamRef.current=null;setCamera(false);};
 React.useEffect(()=>()=>{if(timerRef.current)clearInterval(timerRef.current);streamRef.current?.getTracks().forEach(t=>t.stop());},[]);
 return <main><h2>QR Code Threat Analyzer</h2><p className="sub">Decode QR content safely. KRYPTO LENS never opens the extracted URL.</p>
  <div className="card"><h3>Upload a QR image</h3><input type="file" accept="image/*" onChange={decodeFile}/><button className="btn" onClick={startCamera} disabled={camera}>Start camera scan</button>{camera&&<button className="btn alt" onClick={stopCamera}>Stop camera</button>}
   {camera&&<video ref={videoRef} playsInline muted style={{display:'block',width:'100%',maxWidth:480,marginTop:12,borderRadius:12}}/>}
   {busy&&<p>Analyzing decoded content…</p>}
  </div>
  {result&&!result.error&&<div className="card" style={{marginTop:16}}><span className={'pill '+result.verdict}>{result.verdict}</span><h3>Risk score: {result.score}/100</h3><p>Decoded content type: {result.qrContentType}</p><p style={{overflowWrap:'anywhere'}}>{result.extractedContent}</p><ul className="fl">{(result.flags||[]).map((f,i)=><li key={i}>{f}</li>)}</ul><p><b>Safety:</b> Never open suspicious QR links. Verify the domain independently and avoid entering credentials or OTPs.</p><button className="btn" onClick={()=>downloadThreatPDF(result)}>Download PDF report</button><div className="tabs"><button onClick={()=>speakText(`QR code ${result.qrContentType}. ${result.verdict}. Risk score ${result.score} out of 100. ${(result.flags||[]).join('. ')}. Do not open suspicious links.`)}>▶ Play</button><button onClick={()=>speakText('', 'pause')}>Ⅱ Pause</button><button onClick={()=>speakText('', 'stop')}>■ Stop</button></div></div>}
  {result?.error&&<p className="err">{result.error}</p>}
 </main>;
}

function Password() {
  const [v, setV] = useState(''), [r, setR] = useState(null), [show, setShow] = useState(false);
  useEffect(() => {
    if (!v) { setR(null); return; }
    let active = true;
    const id = setTimeout(async () => {
      const result = await api('scan/password', { data: v });
      if (active) setR(result);
    }, 350);
    return () => { active = false; clearTimeout(id); };
  }, [v]);
  return (<main>
    <h2>Password strength</h2><p className="sub">Checked on your own server. Only the length is saved, never the password.</p>
    <div className="card">
      <input aria-label="Password" type={show ? 'text' : 'password'} placeholder="Type a password to test" value={v} onChange={e => setV(e.target.value)} autoComplete="off" />
      <label><input type="checkbox" style={{ width: 'auto', margin: '0 8px 0 0' }} checked={show} onChange={e => setShow(e.target.checked)} />Show password</label>
    </div>
    <Result r={r} strength />
  </main>);
}

function IP() {
  const [v, setV] = useState(''), [r, setR] = useState(null);
  const run = async () => {
    const endpoint = v.trim() ? 'ip/' + encodeURIComponent(v.trim()) : 'ip';
    setR(await api(endpoint));
  };
  return (<main>
    <h2>IP lookup</h2><p className="sub">Leave empty to look up the server's public IP.</p>
    <div className="card"><input aria-label="IP address" placeholder="8.8.8.8" value={v} onChange={e => setV(e.target.value)} onKeyDown={e => e.key === 'Enter' && run()} />
      <button className="btn" onClick={run}>Look up</button></div>
    {r && (r.error ? <p className="err">{r.error}</p> : <div className="card" style={{ marginTop: 16 }}>
      {Object.entries(r).map(([k, x]) => <div className="kv" key={k}><span>{k}</span><b>{String(x ?? '-')}</b></div>)}</div>)}
  </main>);
}

function ReportPage() {
  const [u, setU] = useState(''), [n, setN] = useState(''), [m, setM] = useState('');
  const send = async () => { const r = await api('report', { url: u, note: n }); setM(r.ok ? 'Thanks. Your report was saved.' : r.error); if (r.ok) { setU(''); setN(''); } };
  return (<main>
    <h2>Report a threat</h2><p className="sub">Tell us about a suspicious link so it can be reviewed.</p>
    <div className="card"><input aria-label="Suspicious link" placeholder="Suspicious link" value={u} onChange={e => setU(e.target.value)} />
      <textarea aria-label="What happened" style={{ minHeight: 90 }} placeholder="Where did you find it? What did it ask for?" value={n} onChange={e => setN(e.target.value)} />
      <button className="btn" disabled={!u.trim()} onClick={send}>Send report</button>
      {m && <p className={m.startsWith('Thanks') ? 'ok' : 'err'}>{m}</p>}</div>
  </main>);
}

function Awareness() {
  const tips = [['Check the sender','Look at the full email address, not just the display name.'],['Hover before you click','Preview the link destination. If it looks off, do not open it.'],['Never share codes','No real bank or service will ask for your OTP or PIN.'],['Use a password manager','Unique, long passwords for every account.'],['Turn on 2-factor sign-in','An authenticator app beats SMS codes.'],['Slow down on urgent messages','Pressure is the main tool of scammers.'],['Update your devices','Install updates; they fix known security holes.'],['Use trusted Wi-Fi','Avoid banking on public networks without a VPN.']];
  return (<main><h2>Cyber awareness</h2><p className="sub">Eight habits that stop most attacks.</p>
    <div className="grid">{tips.map(([a, b]) => <div className="card" key={a}><h3>{a}</h3><p>{b}</p></div>)}</div></main>);
}


function VoiceAssistant(){
 const [question,setQuestion]=useState(''),[answer,setAnswer]=useState(''),[enabled,setEnabled]=useState(true);
 const ask=()=>{
  const q=question.toLowerCase();let a;
  if(!lastScanResult) a={en:'Run an email, URL or QR scan first. I can then explain its risk indicators.',hi:'पहले ईमेल, URL या QR स्कैन करें। फिर मैं जोखिम के संकेत समझा सकता हूँ।',kn:'ಮೊದಲು ಇಮೇಲ್, URL ಅಥವಾ QR ಸ್ಕ್ಯಾನ್ ಮಾಡಿ. ನಂತರ ಅಪಾಯದ ಸೂಚನೆಗಳನ್ನು ವಿವರಿಸುತ್ತೇನೆ.'}[currentLang];
  else {
   const r=lastScanResult, reasons=(r.flags||[]).join('. ');
   a=`${r.verdict}. Risk score ${r.score} out of 100. ${reasons||'No major indicators were detected.'} ${r.verdict==='Safe'?'Stay alert and verify the sender.':'Do not open the link or provide passwords, OTPs or payment details. Verify through the official website or app.'}`;
   if(currentLang==='hi') a=`${r.verdict==='Safe'?'सुरक्षित':r.verdict==='Suspicious'?'संदिग्ध':'खतरनाक'}। जोखिम स्कोर 100 में से ${r.score}। ${reasons}। संदिग्ध लिंक न खोलें और पासवर्ड या OTP साझा न करें।`;
   if(currentLang==='kn') a=`${r.verdict==='Safe'?'ಸುರಕ್ಷಿತ':r.verdict==='Suspicious'?'ಸಂದೇಹಾಸ್ಪದ':'ಅಪಾಯಕಾರಿ'}. ಅಪಾಯದ ಅಂಕ 100ರಲ್ಲಿ ${r.score}. ${reasons}. ಅನುಮಾನಾಸ್ಪದ ಲಿಂಕ್ ತೆರೆಯಬೇಡಿ ಮತ್ತು ಪಾಸ್‌ವರ್ಡ್ ಅಥವಾ OTP ಹಂಚಿಕೊಳ್ಳಬೇಡಿ.`;
  }
  setAnswer(a);if(enabled)speakText(a);
 };
 return <section className="card" style={{margin:'0 auto 24px',maxWidth:1060}}><h3>AI Voice Guidance</h3><label><input type="checkbox" style={{width:'auto',marginRight:8}} checked={enabled} onChange={e=>setEnabled(e.target.checked)}/>Enable voice assistance</label><input value={question} onChange={e=>setQuestion(e.target.value)} placeholder="Ask: Is this email safe? Why is this link suspicious?"/><div className="tabs"><button onClick={()=>startVoiceQuestion(setQuestion)}>🎙 Ask by voice</button><button className="btn" onClick={ask}>Ask / Explain</button><button onClick={()=>speakText(answer)} disabled={!answer}>▶ Play</button><button onClick={()=>speakText('', 'pause')}>Ⅱ Pause</button><button onClick={()=>speakText('', 'stop')}>■ Stop</button></div>{answer&&<p aria-live="polite">{answer}</p>}</section>;
}

function App() {
  const [p, setP] = useState(location.hash.slice(1) || 'home');
  const go = x => { setP(x); location.hash = x; window.scrollTo(0, 0); };
  const View = { home: <Home go={go} />, dashboard: <Dashboard />, scanner: <Scanner />, qr: <QRAnalyzer />, password: <Password />, ip: <IP />, report: <ReportPage />, awareness: <Awareness /> }[p] || <Home go={go} />;
  return (<>
    <nav><span className="logo">KRYPTO LENS</span><label style={{color:'var(--muted)',fontSize:13}}>Language <select aria-label="Language" value={currentLang} onChange={e=>setLanguage(e.target.value)} style={{width:'auto',margin:'0 8px',padding:7,background:'#0a1324',color:'var(--text)',border:'1px solid var(--line)',borderRadius:8}}><option value="en">English</option><option value="kn">ಕನ್ನಡ</option><option value="hi">हिन्दी</option></select></label>{PAGES.map(([k, l]) => <button key={k} className={p === k ? 'on' : ''} onClick={() => go(k)}>{l}</button>)}</nav>
    {View}
    <VoiceAssistant />
  </>);
}
ReactDOM.createRoot(document.getElementById('root')).render(<App />);
setTimeout(translatePage, 0);
</script>
</body>
</html>


Overwriting cybershield/public/index.html


## 6. Install Node.js dependencies

In [43]:
%cd /content/cybershield
!npm install
%cd /content


/content/cybershield
⠙⠹⠸
up to date, audited 71 packages in 560ms
⠸
⠸19 packages are looking for funding
⠸  run `npm fund` for details
⠸
found 0 vulnerabilities
⠸/content


## 7. Test the Python analyzer
The tests use sample data only. They do not contact external services.

In [44]:
import json, subprocess

def run_local_scan(scan_type, value):
    proc = subprocess.run(
        ["python3", "cybershield/analyzer.py"],
        input=json.dumps({"type": scan_type, "data": value}),
        capture_output=True, text=True, check=True
    )
    return json.loads(proc.stdout)

print("URL sample:")
print(json.dumps(run_local_scan("url", "http://paypal-login.verify-account.xyz/signin"), indent=2))
print("\nPassword sample:")
print(json.dumps(run_local_scan("password", "Tr0ub4dor&3-horse-staple"), indent=2))


URL sample:
{
  "score": 55,
  "verdict": "Suspicious",
  "flags": [
    "Connection is not encrypted (no HTTPS)",
    "High-risk top-level domain",
    "Sensitive keywords in URL: login, verify, account, paypal"
  ]
}

Password sample:
{
  "score": 100,
  "verdict": "Strong",
  "flags": [
    "Great password. Store it in a password manager."
  ],
  "entropy": 157.3
}


## 8. Optional API keys

The app works without API keys using its local heuristic checks.

For additional URL reputation checks, optionally add these values to **Colab Secrets** (the key icon):
- `GSB_KEY` — Google Safe Browsing API key
- `VT_KEY` — VirusTotal API key

Do not paste private API keys into a notebook you plan to share. The optional API services have their own quotas and terms.


## 9. Start KRYPTO LENS and display the app
Run this cell once after installing dependencies. The server listens on port 3000. QR camera access depends on browser permission and a secure browser context; if unavailable, use image upload.

In [52]:
import os, subprocess, time, signal
from google.colab import output

# Read optional secrets without failing when they are not configured.
try:
    from google.colab import userdata
    for key in ("GSB_KEY", "VT_KEY"):
        try:
            value = userdata.get(key)
            if value:
                os.environ[key] = value
        except Exception:
            pass
except Exception:
    pass

# Stop an earlier server started by this notebook, if present.
old = subprocess.run(["bash", "-lc", "pkill -f '[n]ode server.js' || true"],
                     capture_output=True, text=True)

server_log_path = "/content/cybershield/server.log"
server_log = open(server_log_path, "w")
server_process = subprocess.Popen(
    ["node", "server.js"],
    cwd="/content/cybershield",
    stdout=server_log,
    stderr=subprocess.STDOUT,
    env=os.environ.copy()
)

# Wait briefly for the server and fail early with its log if startup failed.
for _ in range(30):
    if server_process.poll() is not None:
        server_log.flush()
        print(open(server_log_path).read())
        raise RuntimeError("CyberShield server stopped during startup.")
    try:
        import urllib.request
        with urllib.request.urlopen("http://127.0.0.1:3000/api/dashboard", timeout=2) as response:
            if response.status == 200:
                break
    except Exception:
        time.sleep(1)
else:
    print(open(server_log_path).read())
    raise TimeoutError("CyberShield did not become ready on port 3000.")

server_log.flush()
print("KRYPTO LENS server is ready.")
print(open(server_log_path).read())
output.serve_kernel_port_as_iframe(3000, height=850)

KRYPTO LENS server is ready.
KRYPTO LENS is running on port 3000
Local demo storage: cybershield/data/cybershield-data.json
Optional threat-intelligence APIs: false



<IPython.core.display.Javascript object>

## 10. API smoke tests (run after Section 9)

In [54]:
import json
import urllib.request
import urllib.error
import socket
import time

API_BASE = "http://127.0.0.1:3000"

def get_json(url, timeout=10):
    request = urllib.request.Request(url, method="GET")
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))

def post_json(url, payload, timeout=30):
    data = json.dumps(payload).encode("utf-8")

    request = urllib.request.Request(
        url,
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))

    except urllib.error.HTTPError as e:
        error_body = e.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"HTTP Error {e.code}: {error_body}"
        ) from e

    except urllib.error.URLError as e:
        raise RuntimeError(
            f"Cannot connect to KRYPTO LENS server: {e.reason}. "
            "Run the server startup cell (Section 9) first."
        ) from e


# STEP 1: Check whether the backend is running
print("Checking KRYPTO LENS backend...")

try:
    dashboard = get_json(f"{API_BASE}/api/dashboard")
    print("Backend connection: SUCCESS")

except Exception as e:
    raise RuntimeError(
        f"Backend is unavailable: {e}\n"
        "Execute Section 9 to start the server, then retry."
    )


# STEP 2: Test QR code containing a URL
print("\nTesting QR URL analysis...")

qr_url_result = post_json(
    f"{API_BASE}/api/scan/qr",
    {"data": "https://example.com/login"}
)

assert qr_url_result.get("qrContentType") == "URL", qr_url_result
assert isinstance(qr_url_result.get("score"), (int, float)), qr_url_result
assert 0 <= qr_url_result["score"] <= 100

print("QR URL analysis: PASS")
print("Risk score:", qr_url_result["score"])
print("Risk category:", qr_url_result["verdict"])


# STEP 3: Test QR code containing ordinary text
print("\nTesting QR text analysis...")

qr_text_result = post_json(
    f"{API_BASE}/api/scan/qr",
    {"data": "Hello from a sample QR code"}
)

assert qr_text_result.get("qrContentType") == "Text", qr_text_result
assert isinstance(qr_text_result.get("score"), (int, float)), qr_text_result
assert 0 <= qr_text_result["score"] <= 100

print("QR text analysis: PASS")
print("Risk score:", qr_text_result["score"])
print("Risk category:", qr_text_result["verdict"])


# STEP 4: Verify dashboard statistics
print("\nChecking dashboard statistics...")

dashboard = get_json(f"{API_BASE}/api/dashboard")
stats = dashboard.get("stats", {})

assert "qrs" in stats, stats

print("Total QR scans:", stats["qrs"])
print("Total emails:", stats.get("emails", 0))
print("Total URLs:", stats.get("urls", 0))
print("Safe detections:", stats.get("safe", 0))
print("Suspicious detections:", stats.get("suspicious", 0))
print("Dangerous detections:", stats.get("dangerous", 0))


print("\n================================")
print("KRYPTO LENS API TEST COMPLETED")
print("================================")

Checking KRYPTO LENS backend...
Backend connection: SUCCESS

Testing QR URL analysis...
QR URL analysis: PASS
Risk score: 0
Risk category: Safe

Testing QR text analysis...
QR text analysis: PASS
Risk score: 0
Risk category: Safe

Checking dashboard statistics...
Total QR scans: 2
Total emails: 1
Total URLs: 6
Safe detections: 8
Suspicious detections: 0
Dangerous detections: 1

KRYPTO LENS API TEST COMPLETED


## 12. Troubleshooting
- **`%%writefile` fails:** run notebook cells in Google Colab, not in a Python script or terminal.
- **`npm install` fails:** rerun the install cell and inspect the output.
- **Camera is blocked:** allow camera access for the browser tab; use QR image upload if camera APIs are unavailable.
- **Speech recognition is unavailable:** use a supported Chromium browser or type the question. Text-to-speech support is checked separately.
- **Kannada/Hindi voice is unavailable:** speech synthesis depends on installed browser voices; the browser may fall back to its default voice.
- **PDF download does not start:** check connectivity to the jsPDF CDN and reload the page.
- **External reputation checks:** add `GSB_KEY` and/or `VT_KEY` to Colab Secrets. Without them, local heuristic analysis still works.

## 11. Execution order and feature checks
Run Sections **1 → 8** in order, then **9** to start the backend, then **12** for the API smoke test. Open the website and manually test QR image upload, camera permission, language switching and voice controls. Browser camera/microphone permissions cannot be reliably automated from a notebook test cell.

## 13. Open KRYPTO LENS in a new browser tab

After running the server cell (Section 9), run the cell below. It creates a clickable link to the KRYPTO LENS website through Google Colab's port proxy. The link works while this Colab runtime is running.

In [55]:
from google.colab import output
from IPython.display import display, HTML

# Create the Colab proxy URL for the app running on port 3000.
website_url = output.eval_js("google.colab.kernel.proxyPort(3000)")

display(HTML(
    f'<h3>CyberShield is ready</h3>'
    f'<p><a href="{website_url}" target="_blank" rel="noopener noreferrer" '
    f'style="display:inline-block;padding:12px 18px;background:#0891b2;'
    f'color:white;text-decoration:none;border-radius:8px;font-weight:bold;">'
    f'Open CyberShield Website</a></p>'
    f'<p>Website link: <a href="{website_url}" target="_blank">{website_url}</a></p>'
))